<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/07_Adaptive_Rag_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Langchain/

Mounted at /content/drive
/content/drive/MyDrive/Langchain


In [ ]:
#Python 3.12.13
!pip install -U langchain==1.3.11
!pip install requests==2.34.2
!pip install -U langchain_community==0.4.2
!pip install -U python-dotenv==1.2.2
!pip install -U langchain-huggingface==1.2.2
!pip install faiss-cpu==1.14.3
!pip install langchain-groq==1.1.3
!pip install chromadb==1.5.9
!pip install langchain-classic==1.0.8
!pip install -U langgraph==1.2.7

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS ,Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
from langchain_core.utils.pydantic import BaseModel , Field
from typing import Literal
import os
load_dotenv(find_dotenv())

/tmp/ipykernel_5796/1163035779.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


True

### Router Node

In [ ]:

class Router(BaseModel) :
  source : Literal["vecotor_store" , "web_search"] = Field(... , description= "for given user question the model chooses to route it to web search or vector store ")

system ="""You are an expert in routing a user question to a vector store or web search \n
the vector store contains documents related to  retrieval-augmented-generation-rag , programming fundamentals and what is fintech .
use the vector store to answer questions about these topics and otherwise use web-search """

routing_prompt = ChatPromptTemplate.from_messages([("system" , system) , ("human" , "{question}")])

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structured_routing_llm = llm.with_structured_output(Router)

quesion_router = routing_prompt | structured_routing_llm

### Retrieval node

In [ ]:
docs = WebBaseLoader(["https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag" ,
                      "https://aws.amazon.com/what-is/retrieval-augmented-generation/" ,
                      "https://www.geeksforgeeks.org/computer-science-fundamentals/programming-tutorial/" ,
                      "https://www.geeksforgeeks.org/blogs/what-is-fintech/"]).load()
# spliting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
# embedding modle
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)

#  create vDB
vector_db = Chroma.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Grade document retrieved Node

In [ ]:
class GradeDocument(BaseModel) :

  score : str = Field(... , description= "Documents are relivant to Query :'yes' or 'no' " )

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_matching_grader = llm.with_structured_output(GradeDocument)


system = """You are a grader assessing relevance of a retrieved document to a user question. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human","Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)
retrieval_grader =(grade_prompt | structred_llm_matching_grader )

### Generate Answer Node

In [ ]:
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)

template = "based on this context :\n {context} \n\n answer this question {question}"
prompt = ChatPromptTemplate.from_template(template)


generate_answer = prompt | llm | StrOutputParser()

## Question Re-writer Node

In [ ]:
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)

system = """You a question re-writer that converts an input question to a better version that is optimized \n
     for vectorstore retrieval. Look at the input and try to reason about the underlying sematic intent / meaning. Just say the question. I dont need any explanation just question is enough"""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Here is the initial question: \n\n {question} \n Formulate an improved question."),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()

## Web search Node

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
web_search_tool = TavilySearchResults(k=3)

/tmp/ipykernel_5796/2239647488.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(k=3)


## Hallucination Grader `edge`

In [ ]:
class GradeHallucination(BaseModel):
  score: str = Field(description="Answer is grounded in the facts, 'yes' or 'no'")

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_hallucination = llm.with_structured_output(GradeHallucination)

system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)
hallucination_grader = hallucination_prompt | structred_llm_hallucination

## Answer Grader `edge`

In [ ]:
class GradeAnswer(BaseModel):
  score: str = Field(description="Answer addresses the question, 'yes' or 'no'")

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_answer_grader = llm.with_structured_output(GradeAnswer)


system = """You are a grader assessing whether an answer addresses / resolves a question \n
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structred_llm_answer_grader


# Graph

In [ ]:
# helpers
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
class GraphState(TypedDict) :

  question:str
  documents : List[Document]
  generation :str
#-----------------------------------------------------------------

# define nodes functions

def retrive (state) :
  print("------retrive---node-----")
  question = state["question"]

  documents = retriever.invoke(question)
  return {"documents": documents , "question" : question }
#--------------------------------------------------------
def grade_docs (state) :
  print("---------GradeDocs----node----")
  question =state["question"]
  documents = state["documents"]
  retrived_docs =[]
  for doc in documents:
    score= retrieval_grader.invoke({"document": doc , "question" : question })
    grade = score.score

    if grade =="yes" :
      print("---GRADE: DOCUMENT RELEVANT---")
      retrived_docs.append(doc)
    else :
      print("---GRADE: DOCUMENT NOT RELEVANT---")
  return {"documents" : retrived_docs , "question" : question}
  #--------------------------------------------------------
def generate_answer_node (state) :
  print("---------generate_answer----node----")
  question =state["question"]
  documents = state["documents"]

  generation = generate_answer.invoke({"context": format_docs(documents), "question": question})
  return {"documents" : documents ,"question" : question ,"generation" : generation }

  #-------------------------------------------------------

def re_write_question(state) :

  print("---REWRITE_QUESTION------NODE--")
  question = state["question"]
  documents = state["documents"]

  # Re-write question
  better_question = question_rewriter.invoke({"question": question})
  return {"documents": documents, "question": better_question}


def web_search(state) :
  print("------Web_search----node----")
  question = state["question"]

  documents = web_search_tool.invoke(question)
  web_resaults = "\n".join(doc["content"] for doc in documents)
  web_resaults = [Document(page_content=web_resaults)]
  # print(web_resaults)
  return {"documents": web_resaults , "question": question}

#-----------------------------------------------------------------

# define edges functions

def route_question (state) :
  print("------------ROUTE QUESTION----------")
  question = state["question"]
  route = quesion_router.invoke({"question": question})

  if route.source == "web_search" :
    print("---ROUTE QUESTION TO RAG ---")
    return "web_search"
  elif route.source == "vecotor_store" :
    print("---ROUTE QUESTION TO VECTOR STORE ---")
    return "retrive"
#---------------------------------------------------------
def decide_to_generate (state) :
  print("---ASSESS GRADED DOCUMENTS---")
  question = state["question"]
  filtered_documents = state["documents"]



  if not filtered_documents :
    print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---")
    return "re_write_question"
  else:

  # We have relevant documents, so generate answer
    print("---DECISION: GENERATE---")
    return "generate_answer"
#----------------------------------------------------------
def is_there_hallucination_and_good_answer (state) :
    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]


    score = hallucination_grader.invoke({"documents": documents, "generation": generation})
    grade = score.score

    if grade == "yes" :
      print("-----Check is answer useful --------")
      score = answer_grader.invoke({"question": question,"generation": generation})
      grade = score.score

      if grade == "yes":
        print("---DECISION: GENERATION ADDRESSES QUESTION---")
        return "useful"
      else:
        print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION--- i'm rewriting the question ---")
        return "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"


In [ ]:
from langgraph.graph import StateGraph , END

workflow = StateGraph(GraphState)


workflow.add_node("retrive", retrive)
workflow.add_node("grade_docs", grade_docs)
workflow.add_node("generate_answer", generate_answer_node)
workflow.add_node("re_write_question", re_write_question)
workflow.add_node("web_search", web_search)

workflow.set_conditional_entry_point(route_question , {"retrive" : "retrive" ,
                                                    "web_search" : "web_search"})

workflow.add_edge("web_search" , "generate_answer")
workflow.add_edge("retrive" , "grade_docs")
workflow.add_conditional_edges( "grade_docs", decide_to_generate , {"re_write_question" :"re_write_question" ,
                                                "generate_answer" :"generate_answer"})
workflow.add_edge("re_write_question" , "retrive")
workflow.add_conditional_edges("generate_answer", is_there_hallucination_and_good_answer , {"useful" : END ,
                                                                         "not supported" : "generate_answer" ,
                                                                         "not useful" : "re_write_question"})

app=workflow.compile()


In [ ]:
app.invoke({"question" : "what is the rag ?"})["generation"]

------------ROUTE QUESTION----------
---ROUTE QUESTION TO VECTOR STORE ---
------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
---------generate_answer----node----
---CHECK HALLUCINATIONS---
-----Check is answer useful --------
---DECISION: GENERATION ADDRESSES QUESTION---


"According to the context, **Retrieval-Augmented Generation (RAG)** is the process of optimizing the output of a large language model by referencing an authoritative knowledge base outside of its training data sources before generating a response. In other words, RAG extends the capabilities of Large Language Models (LLMs) to specific domains or an organization's internal knowledge base, allowing for more accurate and relevant output without the need to retrain the model."

In [ ]:
app.invoke({"question" : "what is the fintech ?"})["generation"]

------------ROUTE QUESTION----------
---ROUTE QUESTION TO VECTOR STORE ---
------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
---------generate_answer----node----
---CHECK HALLUCINATIONS---
-----Check is answer useful --------
---DECISION: GENERATION ADDRESSES QUESTION---


'According to the context, FinTech (Financial Technology) is an emerging trend of technology in traditional financial services that provides various types of software used for financial products and services. It is an innovative creation of the 21st century that helps consumers and businesses in financial transactions through digital delivery of financial products and services, allowing for fast, easy, and secure transactions without the need to visit bank branches physically.'

In [ ]:
app.invoke({"question" : "how to lose weight"})["generation"]

------------ROUTE QUESTION----------
---ROUTE QUESTION TO RAG ---
------Web_search----node----
---------generate_answer----node----
---CHECK HALLUCINATIONS---
-----Check is answer useful --------
---DECISION: GENERATION ADDRESSES QUESTION---


"To lose weight, follow these steps:\n\n1. **Eat a healthy diet**:\n\t* Eat at least 4 servings of vegetables and 3 servings of fruits a day.\n\t* Choose whole grains, such as brown rice, barley, and whole-wheat bread and pasta.\n\t* Use healthy fats, such as olive oil, vegetable oils, avocados, nuts, and nut butters.\n\t* Limit foods and drinks with added sugar.\n\t* Choose low-fat or fat-free dairy products.\n2. **Stay hydrated**:\n\t* Drink plenty of water throughout the day.\n\t* Limit sugary drinks.\n3. **Exercise regularly**:\n\t* Aim for at least 30 minutes of aerobic exercise, such as brisk walking, most days of the week.\n\t* Incorporate strength training exercises at least twice a week.\n\t* Increase physical activity by taking the stairs, parking further away, and standing instead of sitting.\n4. **Get enough sleep**:\n\t* Aim for 7-9 hours of sleep per night.\n\t* Establish a consistent sleep schedule.\n5. **Be mindful of your eating habits**:\n\t* Eat slowly and savor your